In [1]:
import numpy as np
import math
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ============================================================
# 1. INPUT DATA
# ============================================================

X = np.array([
    [0.66579958, 0.12396913],
    [0.87779099, 0.7786275 ],
    [0.14269907, 0.34900513],
    [0.84527543, 0.71112027],
    [0.45464714, 0.29045518],
    [0.57771284, 0.77197318],
    [0.43816606, 0.68501826],
    [0.34174959, 0.02869772],
    [0.33864816, 0.21386725],
    [0.70263656, 0.9265642],
    [0.926564, 1.026564],
    [0.747504, 0.20897],
    [0.683406, 0.063769],
    [0.583137, 0.012549],
    [0.991457, 0.001744],
    [0.991457, 0.001744]
])

y = np.array([
    0.53899612, 0.42058624, -0.06562362, 0.29399291, 0.21496451,
    0.02310555, 0.24461934, 0.03874902, -0.01385762, 0.61120522,
    -0.04199554, 0.28033031, 0.62973064, 0.06695726, 0.11670354823827367,
    0.11353912028668156
])

# Clip to [0,1]
X = np.clip(X, 0, 1)

# ============================================================
# 2. MODEL SELECTION (REFINED ARCHITECTURE)
# ============================================================

class NoiseInjector:
    """Injects Gaussian noise into features during training to mimic dropout."""
    def __init__(self, sigma=0.01, active=True):
        self.sigma = sigma
        self.active = active
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        if self.active:
            return X + np.random.normal(0, self.sigma, X.shape)
        return X


def make_model(seed):
    """
    Improved small-network architecture:
    - Deepens the MLP: (64, 32, 16)
    - Adaptive learning + early stopping
    - Small L2 penalty
    - NoiseInjector acts like dropout
    """
    return Pipeline([
        ("scaler", StandardScaler()),
        ("noise", NoiseInjector(sigma=0.015, active=True)),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(64, 32, 16),
            activation='relu',
            solver='adam',
            learning_rate='adaptive',
            learning_rate_init=0.01,
            max_iter=5000,
            tol=1e-7,
            alpha=0.0005,
            early_stopping=True,
            n_iter_no_change=30,
            random_state=seed
        ))
    ])


def fit_ensemble(X, y, n=12):
    """Deeper ensemble (n=12 improves uncertainty quality)."""
    models = []
    for i in range(n):
        model = make_model(300 + i)
        model.fit(X, y)
        models.append(model)
    return models


def ensemble_predict(models, Xcand):
    preds = np.vstack([m.predict(Xcand) for m in models])
    mu = preds.mean(axis=0)
    std = preds.std(axis=0, ddof=1) + 1e-9
    return mu, std


# ============================================================
# 3. EI & PI UTILITIES
# ============================================================

def erf_vec(x):
    return np.vectorize(math.erf)(x)

def compute_pi_ei(mu, std, y_best, xi=0.01):
    z = (mu - y_best - xi) / std
    pdf = (1 / np.sqrt(2*np.pi)) * np.exp(-0.5 * z*z)
    cdf = 0.5 * (1 + erf_vec(z / np.sqrt(2)))
    ei = (mu - y_best - xi) * cdf + std * pdf
    ei[std <= 0] = 0
    return cdf, ei


# ============================================================
# 4. TRAIN THE ENSEMBLE
# ============================================================

models = fit_ensemble(X, y, n=12)

idx_best = np.argmax(y)
x_best = X[idx_best]
y_best = y[idx_best]


# ============================================================
# 5. CANDIDATE GENERATION
# ============================================================

rng = np.random.default_rng(999)

N = 25000
Xcand = rng.uniform(0, 1, size=(N, 2))

# Local exploration region
local = rng.normal(loc=x_best, scale=0.04, size=(6000, 2))
local = np.clip(local, 0, 1)
Xcand = np.vstack([Xcand, local])


# ============================================================
# 6. EVALUATE EI/PI
# ============================================================

mu, std = ensemble_predict(models, Xcand)
pi, ei = compute_pi_ei(mu, std, y_best=y_best)

k = int(np.argmax(ei))
x_next = Xcand[k]
mu_next = mu[k]
std_next = std[k]
pi_next = pi[k]
ei_next = ei[k]


# ============================================================
# 7. OUTPUT
# ============================================================

print("================================================")
print("CURRENT BEST OBSERVED")
print("================================================")
print(f"x_best = {x_best}, y_best = {y_best:.6f}\n")

print("================================================")
print("RECOMMENDED NEXT POINT (EI)")
print("================================================")
print(f"x_next = {x_next}")
print(f"μ(x_next)  = {mu_next:.6f}")
print(f"σ(x_next)  = {std_next:.6f}")
print(f"PI         = {pi_next:.4f}")
print(f"EI         = {ei_next:.6f}\n")

print("================================================")
print("REASONING (Model Refinement Summary)")
print("================================================")
print(f"""
• Architecture refined: (64 → 32 → 16) hidden layers
• Added Gaussian NoiseInjector = dropout-like effect
• Early stopping for better generalisation
• Increased ensemble to 12 networks

• EI found the next point balancing:
      – high predicted mean: {mu_next:.6f}
      – uncertainty:         {std_next:.6f}
      – improvement chance:  {pi_next:.2f}

• Domain strictly enforced: X ∈ [0,1]²

• This refined ensemble produces smoother, more reliable
  uncertainty → more stable EI search → better BBO performance.
""")


CURRENT BEST OBSERVED
x_best = [0.683406 0.063769], y_best = 0.629731

RECOMMENDED NEXT POINT (EI)
x_next = [0.99145752 0.0017445 ]
μ(x_next)  = 0.584517
σ(x_next)  = 0.237729
PI         = 0.4082
EI         = 0.069780

REASONING (Model Refinement Summary)

• Architecture refined: (64 → 32 → 16) hidden layers
• Added Gaussian NoiseInjector = dropout-like effect
• Early stopping for better generalisation
• Increased ensemble to 12 networks

• EI found the next point balancing:
      – high predicted mean: 0.584517
      – uncertainty:         0.237729
      – improvement chance:  0.41

• Domain strictly enforced: X ∈ [0,1]²

• This refined ensemble produces smoother, more reliable
  uncertainty → more stable EI search → better BBO performance.

